# IPLytics: Comprehensive IPL Analytics (2008-2025)
### A Production-Grade Sports Analytics Project

> **Objective:** Analyze 1,100+ IPL matches across 18 seasons to derive actionable insights on player performance, team strategies, venue dynamics, toss impact, and the revolutionary Impact Player rule introduced in 2023.

---
**Author:** Aaryan Diwan  
**Tech Stack:** Python | Pandas | NumPy | Plotly | Streamlit  
**Data Source:** Cricsheet.org (Ball-by-ball granularity)


In [ ]:
# ============================================================
# 1. SETUP & IMPORTS
# ============================================================
import pandas as pd
import numpy as np
import plotly.express as px
import plotly.graph_objects as go
from plotly.subplots import make_subplots
import warnings
warnings.filterwarnings('ignore')

import plotly.io as pio
pio.templates.default = "plotly_dark"
print("Libraries loaded successfully.")


## 2. Data Ingestion & Preprocessing
Loading the master ball-by-ball dataset covering **all IPL seasons from 2008 to 2025**.

In [ ]:
df = pd.read_csv('../data/all_deliveries.csv')
df['season'] = df['season'].astype(str).str[:4].astype(int)
df = df[df['season'] >= 2008].copy()

print(f"Total Deliveries: {len(df):,}")
print(f"Total Matches:    {df['match_id'].nunique():,}")
print(f"Total Seasons:    {df['season'].nunique()}")
print(f"Season Range:     {df['season'].min()} - {df['season'].max()}")
print(f"Unique Batters:   {df['striker'].nunique()}")
print(f"Unique Bowlers:   {df['bowler'].nunique()}")
print(f"Unique Teams:     {df['batting_team'].nunique()}")
df.head()


## 3. Feature Engineering
Creating derived analytical columns that power all downstream KPIs.

In [ ]:
df['total_runs'] = df['runs_off_bat'] + df['extras']
df['is_boundary'] = df['runs_off_bat'].isin([4, 6]).astype(int)
df['is_four'] = (df['runs_off_bat'] == 4).astype(int)
df['is_six'] = (df['runs_off_bat'] == 6).astype(int)
df['is_dot'] = (df['total_runs'] == 0).astype(int)
df['is_wicket'] = df['wicket_type'].notna().astype(int)
df['is_wide'] = df['wides'].notna().astype(int)
df['is_noball'] = df['noballs'].notna().astype(int)
df['is_legal'] = (~df['wides'].notna() & ~df['noballs'].notna()).astype(int)
df['era'] = df['season'].apply(lambda x: 'Post-Impact Player (2023-25)' if x >= 2023 else 'Pre-Impact Player (2008-22)')
print(f"Feature engineering complete. Columns: {len(df.columns)}")
print(df.groupby('era')['match_id'].nunique())


---
## 4. Tournament Overview Dashboard

In [ ]:
matches_per_season = df.groupby('season')['match_id'].nunique().reset_index()
matches_per_season.columns = ['Season', 'Matches']
fig = px.bar(matches_per_season, x='Season', y='Matches',
             title='Number of Matches Per Season', color='Matches',
             color_continuous_scale='viridis', text='Matches')
fig.update_traces(textposition='outside')
fig.update_layout(height=500, xaxis=dict(dtick=1))
fig.show()


---
## 5. Batting Performance Analytics
Comprehensive evaluation using: Total Runs, Strike Rate, Boundary %, Avg Runs/Match, Consistency.

In [ ]:
batting = df.groupby('striker').agg(
    total_runs=('runs_off_bat', 'sum'), balls_faced=('is_legal', 'sum'),
    fours=('is_four', 'sum'), sixes=('is_six', 'sum'),
    boundaries=('is_boundary', 'sum'), dots_faced=('is_dot', 'sum'),
    matches=('match_id', 'nunique')
).reset_index()
batting = batting[batting['balls_faced'] >= 500].copy()
batting['strike_rate'] = (batting['total_runs'] / batting['balls_faced'] * 100).round(2)
batting['boundary_pct'] = (batting['boundaries'] / batting['balls_faced'] * 100).round(2)
batting['avg_runs_per_match'] = (batting['total_runs'] / batting['matches']).round(2)
batting['dot_ball_pct'] = (batting['dots_faced'] / batting['balls_faced'] * 100).round(2)
batting = batting.sort_values('total_runs', ascending=False)
print(f"Qualified batters (500+ balls): {len(batting)}")
batting.head(10)


In [ ]:
top10 = batting.head(10).sort_values('total_runs')
fig = px.bar(top10, x='total_runs', y='striker', orientation='h',
             title='Top 10 All-Time IPL Run Scorers (2008-2025)',
             color='strike_rate', color_continuous_scale='RdYlGn', text='total_runs',
             labels={'striker':'Batter','total_runs':'Total Runs','strike_rate':'Strike Rate'})
fig.update_traces(textposition='outside')
fig.update_layout(height=500, yaxis={'categoryorder':'total ascending'})
fig.show()


In [ ]:
top10_sr = batting.nlargest(10, 'strike_rate').sort_values('strike_rate')
fig = px.bar(top10_sr, x='strike_rate', y='striker', orientation='h',
             title='Top 10 Highest Strike Rates (Min 500 Balls)',
             color='total_runs', color_continuous_scale='plasma', text='strike_rate',
             labels={'striker':'Batter','strike_rate':'Strike Rate'})
fig.update_traces(textposition='outside')
fig.update_layout(height=500)
fig.show()


In [ ]:
fig = px.scatter(batting, x='avg_runs_per_match', y='strike_rate',
                 size='total_runs', color='boundary_pct', hover_name='striker',
                 title='Batting Efficiency Matrix: Strike Rate vs Avg Runs/Match',
                 labels={'avg_runs_per_match':'Avg Runs/Match','strike_rate':'Strike Rate',
                         'boundary_pct':'Boundary %'}, color_continuous_scale='turbo')
fig.update_layout(height=600)
fig.show()


In [ ]:
top10_bp = batting.nlargest(10, 'boundary_pct').sort_values('boundary_pct')
fig = px.bar(top10_bp, x='boundary_pct', y='striker', orientation='h',
             title='Top 10 Highest Boundary % (Min 500 Balls)',
             color='sixes', color_continuous_scale='hot', text='boundary_pct',
             labels={'striker':'Batter','boundary_pct':'Boundary %','sixes':'Total Sixes'})
fig.update_traces(textposition='outside')
fig.update_layout(height=500)
fig.show()


---
## 6. Bowling Performance Analytics
Evaluating bowlers on Wickets, Economy, Dot Ball %, and Bowling Strike Rate.

In [ ]:
bowling = df.groupby('bowler').agg(
    balls_bowled=('is_legal', 'sum'), runs_conceded=('total_runs', 'sum'),
    wickets=('is_wicket', 'sum'), dots=('is_dot', 'sum'),
    fours_conceded=('is_four', 'sum'), sixes_conceded=('is_six', 'sum'),
    matches=('match_id', 'nunique')
).reset_index()
bowling = bowling[bowling['balls_bowled'] >= 300].copy()
bowling['overs'] = (bowling['balls_bowled'] / 6).round(1)
bowling['economy'] = (bowling['runs_conceded'] / bowling['overs']).round(2)
bowling['bowling_sr'] = (bowling['balls_bowled'] / bowling['wickets'].replace(0, np.nan)).round(2)
bowling['dot_ball_pct'] = (bowling['dots'] / bowling['balls_bowled'] * 100).round(2)
bowling['avg'] = (bowling['runs_conceded'] / bowling['wickets'].replace(0, np.nan)).round(2)
bowling = bowling.sort_values('wickets', ascending=False)
print(f"Qualified bowlers (300+ balls): {len(bowling)}")
bowling.head(10)


In [ ]:
top10_wkts = bowling.head(10).sort_values('wickets')
fig = px.bar(top10_wkts, x='wickets', y='bowler', orientation='h',
             title='Top 10 All-Time IPL Wicket Takers (2008-2025)',
             color='economy', color_continuous_scale='RdYlGn_r', text='wickets',
             labels={'bowler':'Bowler','wickets':'Wickets','economy':'Economy'})
fig.update_traces(textposition='outside')
fig.update_layout(height=500)
fig.show()


In [ ]:
top10_eco = bowling.nsmallest(10, 'economy').sort_values('economy', ascending=False)
fig = px.bar(top10_eco, x='economy', y='bowler', orientation='h',
             title='Top 10 Best Economy Rates (Min 300 Balls)',
             color='dot_ball_pct', color_continuous_scale='viridis', text='economy',
             labels={'bowler':'Bowler','economy':'Economy Rate'})
fig.update_traces(textposition='outside')
fig.update_layout(height=500)
fig.show()


In [ ]:
fig = px.scatter(bowling, x='wickets', y='economy',
                 size='balls_bowled', color='dot_ball_pct', hover_name='bowler',
                 title='Bowling Efficiency Matrix: Economy vs Wickets',
                 labels={'wickets':'Total Wickets','economy':'Economy Rate',
                         'dot_ball_pct':'Dot Ball %'}, color_continuous_scale='turbo')
fig.update_layout(height=600)
fig.show()


---
## 7. Team Performance & Win Analytics
Franchise-level performance metrics: Win rates, dominance patterns.

In [ ]:
innings_totals = df.groupby(['match_id','innings','batting_team']).agg(
    runs=('total_runs','sum'), wickets=('is_wicket','sum'), balls=('is_legal','sum')
).reset_index()

season_map = df.groupby('match_id')['season'].first()

match_scores = innings_totals.pivot_table(index='match_id', columns='innings', values='runs', aggfunc='sum').reset_index()
match_scores = match_scores.rename(columns={1: 'inn1_runs', 2: 'inn2_runs'})
team_inn = innings_totals.groupby(['match_id','innings'])['batting_team'].first().unstack()
team_inn = team_inn.rename(columns={1: 'team_bat_first', 2: 'team_bat_second'})
match_results = match_scores[['match_id', 'inn1_runs', 'inn2_runs']].merge(team_inn[['team_bat_first', 'team_bat_second']], on='match_id').dropna()
match_results['winner'] = match_results.apply(
    lambda r: r['team_bat_first'] if r['inn1_runs'] > r['inn2_runs']
              else r['team_bat_second'] if r['inn2_runs'] > r['inn1_runs']
              else 'Tie', axis=1)
match_results['season'] = match_results['match_id'].map(season_map)

all_teams = pd.concat([match_results['team_bat_first'], match_results['team_bat_second']]).unique()
team_stats = []
for team in all_teams:
    played = len(match_results[(match_results['team_bat_first']==team)|(match_results['team_bat_second']==team)])
    won = len(match_results[match_results['winner']==team])
    if played > 20:
        team_stats.append({'Team':team,'Played':played,'Won':won,'Win%':round(won/played*100,2)})
team_df = pd.DataFrame(team_stats).sort_values('Win%', ascending=False)

fig = px.bar(team_df, x='Win%', y='Team', orientation='h',
             title='All-Time IPL Win Percentage by Franchise',
             color='Win%', color_continuous_scale='RdYlGn', text='Win%')
fig.update_traces(texttemplate='%{text:.1f}%', textposition='outside')
fig.update_layout(height=600, yaxis={'categoryorder':'total ascending'})
fig.show()


---
## 8. Toss Decision Analysis
Does batting first or fielding first matter? And has this changed over time?

In [ ]:
bf_by_season = match_results.groupby('season').apply(
    lambda g: (g['winner'] == g['team_bat_first']).sum() / len(g) * 100
).reset_index()
bf_by_season.columns = ['Season','Bat First Win %']
bf_by_season['Field First Win %'] = 100 - bf_by_season['Bat First Win %']

fig = go.Figure()
fig.add_trace(go.Scatter(x=bf_by_season['Season'], y=bf_by_season['Bat First Win %'],
              mode='lines+markers', name='Bat First Win %', line=dict(color='#FFD700', width=3)))
fig.add_trace(go.Scatter(x=bf_by_season['Season'], y=bf_by_season['Field First Win %'],
              mode='lines+markers', name='Field First Win %', line=dict(color='#00BFFF', width=3)))
fig.add_hline(y=50, line_dash="dash", line_color="gray", opacity=0.5)
fig.update_layout(title='Bat First vs Field First Win % by Season',
                  xaxis_title='Season', yaxis_title='Win %', height=500, xaxis=dict(dtick=1))
fig.show()


---
## 9. Venue Intelligence
Highest-scoring and most bowler-friendly venues across IPL history.

In [ ]:
first_innings = innings_totals[innings_totals['innings']==1].copy()
venue_map = df.groupby('match_id')['venue'].first()
first_innings['venue'] = first_innings['match_id'].map(venue_map)
venue_avg = first_innings.groupby('venue').agg(avg_score=('runs','mean'), matches=('match_id','nunique')).reset_index()
venue_avg = venue_avg[venue_avg['matches'] >= 15]
venue_avg['avg_score'] = venue_avg['avg_score'].round(1)

top_v = venue_avg.nlargest(10, 'avg_score').sort_values('avg_score')
fig = px.bar(top_v, x='avg_score', y='venue', orientation='h',
             title='Top 10 Highest-Scoring Venues (Avg 1st Innings, Min 15 Matches)',
             color='avg_score', color_continuous_scale='hot', text='avg_score',
             labels={'venue':'Venue','avg_score':'Avg Score'})
fig.update_traces(textposition='outside')
fig.update_layout(height=500, yaxis={'categoryorder':'total ascending'})
fig.show()


In [ ]:
low_v = venue_avg.nsmallest(10, 'avg_score').sort_values('avg_score', ascending=False)
fig = px.bar(low_v, x='avg_score', y='venue', orientation='h',
             title='Top 10 Most Bowler-Friendly Venues',
             color='avg_score', color_continuous_scale='viridis', text='avg_score',
             labels={'venue':'Venue','avg_score':'Avg Score'})
fig.update_traces(textposition='outside')
fig.update_layout(height=500, yaxis={'categoryorder':'total descending'})
fig.show()


---
## 10. Season-Over-Season Scoring Trends
Tracking the dramatic post-2023 scoring explosion.

In [ ]:
season_rpo = df.groupby('season').agg(total_runs=('total_runs','sum'), legal_balls=('is_legal','sum')).reset_index()
season_rpo['overs'] = season_rpo['legal_balls'] / 6
season_rpo['runs_per_over'] = (season_rpo['total_runs'] / season_rpo['overs']).round(2)

fig = px.line(season_rpo, x='season', y='runs_per_over',
              title='Average Runs Per Over by Season (The Scoring Explosion)',
              markers=True, text='runs_per_over', labels={'season':'Season','runs_per_over':'RPO'})
fig.add_vrect(x0=2022.5, x1=season_rpo['season'].max()+0.5,
              fillcolor="red", opacity=0.1, line_width=0,
              annotation_text="Impact Player Era", annotation_position="top left")
fig.update_traces(textposition='top center', line=dict(width=3, color='#FFD700'))
fig.update_layout(height=500, xaxis=dict(dtick=1))
fig.show()


In [ ]:
innings_season = innings_totals.copy()
innings_season['season'] = innings_season['match_id'].map(season_map)
innings_season = innings_season[innings_season['innings'].isin([1,2])]
total_inn = innings_season.groupby('season').size().reset_index(name='total')
above_200 = innings_season[innings_season['runs']>=200].groupby('season').size().reset_index(name='cnt200')
s200 = total_inn.merge(above_200, on='season', how='left').fillna(0)
s200['pct'] = (s200['cnt200'] / s200['total'] * 100).round(2)

fig = px.bar(s200, x='season', y='pct', title='% of Innings Scoring 200+ by Season',
             color='pct', color_continuous_scale='reds', text='pct',
             labels={'season':'Season','pct':'% 200+ Innings'})
fig.add_vrect(x0=2022.5, x1=s200['season'].max()+0.5, fillcolor="red", opacity=0.1, line_width=0)
fig.update_traces(texttemplate='%{text:.1f}%', textposition='outside')
fig.update_layout(height=500, xaxis=dict(dtick=1))
fig.show()


In [ ]:
sixes_season = df.groupby('season')['is_six'].sum().reset_index()
sixes_season.columns = ['Season','Total Sixes']
fig = px.bar(sixes_season, x='Season', y='Total Sixes', title='Total Sixes Hit Per Season',
             color='Total Sixes', color_continuous_scale='plasma', text='Total Sixes')
fig.update_traces(textposition='outside')
fig.update_layout(height=500, xaxis=dict(dtick=1))
fig.show()


---
## 11. Impact Player Rule Analysis (2023-2025) 
> **The Impact Player rule**, introduced in IPL 2023, allows each team to use a tactical substitute during the match. This section quantifies the rule's statistical effect on scoring patterns, team totals, and match dynamics.

This is the **most analytically relevant section** for 2026 interviews.


In [ ]:
first_innings_era = innings_totals[innings_totals['innings']==1].copy()
first_innings_era['season'] = first_innings_era['match_id'].map(season_map)
first_innings_era['era'] = first_innings_era['season'].apply(
    lambda x: 'Post-Impact Player\n(2023-25)' if x >= 2023 else 'Pre-Impact Player\n(2008-22)')

era_avg = first_innings_era.groupby('era')['runs'].mean().reset_index()
era_avg['runs'] = era_avg['runs'].round(1)

fig = px.bar(era_avg, x='era', y='runs', title='Impact Player Effect: Avg 1st Innings Score',
             color='era', color_discrete_sequence=['#636EFA','#EF553B'], text='runs',
             labels={'era':'Era','runs':'Avg Score'})
fig.update_traces(textposition='outside', textfont_size=16)
fig.update_layout(height=500, showlegend=False)
fig.show()


In [ ]:
fig = px.histogram(first_innings_era, x='runs', color='era',
                   barmode='overlay', nbins=40, opacity=0.7,
                   title='Distribution of 1st Innings Totals: Pre vs Post Impact Player',
                   labels={'runs':'1st Innings Score','era':'Era'},
                   color_discrete_sequence=['#636EFA','#EF553B'])
fig.add_vline(x=200, line_dash="dash", line_color="yellow",
              annotation_text="200 Run Mark", annotation_position="top right")
fig.update_layout(height=500)
fig.show()


In [ ]:
boundary_era = df.groupby('era').agg(
    total_boundaries=('is_boundary','sum'), total_balls=('is_legal','sum'),
    total_sixes=('is_six','sum'), total_fours=('is_four','sum'), total_dots=('is_dot','sum')
).reset_index()
boundary_era['boundary_rate'] = (boundary_era['total_boundaries'] / boundary_era['total_balls'] * 100).round(2)
boundary_era['six_rate'] = (boundary_era['total_sixes'] / boundary_era['total_balls'] * 100).round(2)
boundary_era['dot_rate'] = (boundary_era['total_dots'] / boundary_era['total_balls'] * 100).round(2)

fig = make_subplots(rows=1, cols=3, subplot_titles=['Boundary Rate %','Six Rate %','Dot Ball Rate %'])
fig.add_trace(go.Bar(x=boundary_era['era'], y=boundary_era['boundary_rate'],
              text=boundary_era['boundary_rate'], textposition='outside',
              marker_color=['#636EFA','#EF553B']), row=1, col=1)
fig.add_trace(go.Bar(x=boundary_era['era'], y=boundary_era['six_rate'],
              text=boundary_era['six_rate'], textposition='outside',
              marker_color=['#636EFA','#EF553B']), row=1, col=2)
fig.add_trace(go.Bar(x=boundary_era['era'], y=boundary_era['dot_rate'],
              text=boundary_era['dot_rate'], textposition='outside',
              marker_color=['#636EFA','#EF553B']), row=1, col=3)
fig.update_layout(title='Impact Player Effect on Match Dynamics', height=500, showlegend=False)
fig.show()


In [ ]:
season_rpo_era = season_rpo.copy()
season_rpo_era['era'] = season_rpo_era['season'].apply(
    lambda x: 'Post-Impact Player' if x >= 2023 else 'Pre-Impact Player')
fig = px.bar(season_rpo_era, x='season', y='runs_per_over', color='era',
             title='Runs Per Over by Season (Impact Player Era Highlighted)',
             color_discrete_sequence=['#636EFA','#EF553B'], text='runs_per_over',
             labels={'season':'Season','runs_per_over':'RPO'})
fig.update_traces(texttemplate='%{text:.2f}', textposition='outside')
fig.update_layout(height=500, xaxis=dict(dtick=1))
fig.show()


---
## 12. Executive Summary & Data Export

In [ ]:
with pd.ExcelWriter('../data/IPLytics_Analysis_2025.xlsx', engine='openpyxl') as writer:
    batting.to_excel(writer, sheet_name='Batting KPIs', index=False)
    bowling.to_excel(writer, sheet_name='Bowling KPIs', index=False)
    team_df.to_excel(writer, sheet_name='Team Performance', index=False)
    venue_avg.sort_values('avg_score', ascending=False).to_excel(writer, sheet_name='Venue Stats', index=False)
    s200.to_excel(writer, sheet_name='200+ Trend', index=False)
    boundary_era.to_excel(writer, sheet_name='Impact Player Analysis', index=False)

print("All KPIs exported to: ../data/IPLytics_Analysis_2025.xlsx")
print("\n" + "="*60)
print("  IPLytics Analysis Complete!")
print(f"  Matches Analyzed: {df['match_id'].nunique():,}")
print(f"  Deliveries Processed: {len(df):,}")
print(f"  Seasons Covered: {df['season'].min()} - {df['season'].max()}")
print("="*60)
